# PitVQA → SAGE Training (Robust Version)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matheus-rech/pitvqa-surgical-workflow/blob/main/notebooks/04_pitvqa_sage_robust.ipynb)

**This notebook has robust error handling and verification at each step.**

## What it does:
1. Downloads PitVQA videos.zip (~8GB with pre-extracted frames)
2. Extracts and verifies frames exist
3. Creates SFT dataset with **ACTUAL EMBEDDED IMAGES**
4. Pushes to HuggingFace Hub

## Requirements:
- HuggingFace token with **WRITE** permissions
- ~15GB free disk space
- ~30-60 minutes runtime

---

In [ ]:
#@title 1. Install Dependencies
print("Installing required packages...")
!pip install -q huggingface_hub datasets pillow tqdm numpy requests
!apt-get install -y aria2 > /dev/null 2>&1 || echo "aria2 not available, will use requests"
print("Done!")

In [ ]:
#@title 2. Configuration - ENTER YOUR TOKEN HERE
import os
import shutil

# ============================================
# IMPORTANT: Enter your HuggingFace token below
# Get one at: https://huggingface.co/settings/tokens
# Make sure it has WRITE permissions!
# ============================================

HF_TOKEN = ""  #@param {type:"string"}
HF_USERNAME = "mmrech"  #@param {type:"string"}

# Dataset settings
MAX_FRAMES = 5000  #@param {type:"integer"}
DATASET_NAME = f"{HF_USERNAME}/pitvqa-sage-sft"

# Paths
BASE_DIR = "/content" if os.path.exists("/content") else os.getcwd()
DOWNLOAD_DIR = f"{BASE_DIR}/pitvqa_download"
DATA_DIR = f"{BASE_DIR}/pitvqa_data"

# Create directories
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Check disk space
total, used, free = shutil.disk_usage(BASE_DIR)
free_gb = free // (1024**3)

print("="*50)
print("CONFIGURATION")
print("="*50)
print(f"HF Token: {'SET' if HF_TOKEN else 'NOT SET - PLEASE ENTER ABOVE!'}")
print(f"HF Username: {HF_USERNAME}")
print(f"Dataset: {DATASET_NAME}")
print(f"Max frames: {MAX_FRAMES}")
print(f"Free disk: {free_gb} GB")
print("="*50)

if not HF_TOKEN:
    print("\n" + "!"*50)
    print("WARNING: HF_TOKEN is empty!")
    print("Please enter your token in the field above and re-run this cell.")
    print("!"*50)
elif free_gb < 15:
    print(f"\nWARNING: Only {free_gb}GB free. Need ~15GB.")
else:
    print("\nConfiguration OK! Proceed to next cell.")

In [ ]:
#@title 3. Login to HuggingFace
from huggingface_hub import login, whoami

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is not set! Go back to cell 2 and enter your token.")

print("Logging in to HuggingFace...")
login(token=HF_TOKEN)

# Verify login
try:
    user_info = whoami()
    print(f"Logged in as: {user_info['name']}")
    print("Login successful!")
except Exception as e:
    print(f"Login failed: {e}")
    raise

In [ ]:
#@title 4. Download PitVQA Dataset (~8GB, ~15-30 min)
import subprocess
import requests
from tqdm.auto import tqdm

VIDEOS_URL = "https://rdr.ucl.ac.uk/ndownloader/files/49158880"
VIDEOS_PATH = f"{DOWNLOAD_DIR}/videos.zip"

def download_with_progress(url, output_path):
    """Download file with progress bar and resume support."""
    
    # Check if already downloaded
    if os.path.exists(output_path):
        size_gb = os.path.getsize(output_path) / 1e9
        if size_gb > 7.5:  # Expected ~8GB
            print(f"File already exists: {size_gb:.2f} GB")
            return True
        else:
            print(f"Incomplete file found ({size_gb:.2f} GB), re-downloading...")
            os.remove(output_path)
    
    # Try aria2c first (faster, supports resume)
    try:
        print("Downloading with aria2c (fast, multi-threaded)...")
        result = subprocess.run([
            "aria2c", "-x", "16", "-s", "16", "-k", "1M",
            "--continue=true", "--max-tries=10",
            "-d", os.path.dirname(output_path),
            "-o", os.path.basename(output_path),
            url
        ], capture_output=True, timeout=7200)  # 2 hour timeout
        
        if result.returncode == 0 and os.path.exists(output_path):
            size_gb = os.path.getsize(output_path) / 1e9
            print(f"Download complete: {size_gb:.2f} GB")
            return True
    except Exception as e:
        print(f"aria2c failed: {e}, falling back to requests...")
    
    # Fallback to requests
    print("Downloading with requests (slower but reliable)...")
    response = requests.get(url, stream=True, timeout=7200)
    total = int(response.headers.get('content-length', 0))
    
    with open(output_path, 'wb') as f:
        with tqdm(total=total, unit='B', unit_scale=True, desc="Downloading") as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))
    
    return os.path.exists(output_path)

print("="*50)
print("DOWNLOADING PITVQA DATASET")
print("="*50)
print(f"URL: {VIDEOS_URL}")
print(f"Destination: {VIDEOS_PATH}")
print("This may take 15-30 minutes...\n")

success = download_with_progress(VIDEOS_URL, VIDEOS_PATH)

if success:
    size_gb = os.path.getsize(VIDEOS_PATH) / 1e9
    print(f"\nDownload successful: {size_gb:.2f} GB")
else:
    raise RuntimeError("Download failed!")

In [ ]:
#@title 5. Verify and Extract Frames
import zipfile
from pathlib import Path

print("="*50)
print("EXTRACTING FRAMES")
print("="*50)

# Verify ZIP file
print("\n1. Verifying ZIP file...")
try:
    with zipfile.ZipFile(VIDEOS_PATH, 'r') as zf:
        bad_file = zf.testzip()
        if bad_file:
            raise RuntimeError(f"Corrupt file in ZIP: {bad_file}")
        
        # Count image files
        all_files = zf.namelist()
        image_files = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"   ZIP contains {len(image_files)} image files")
        print(f"   ZIP is valid!")
except zipfile.BadZipFile:
    raise RuntimeError("Invalid ZIP file! Please re-download.")

# Check if already extracted
existing_frames = list(Path(DATA_DIR).rglob("*.jpg")) + list(Path(DATA_DIR).rglob("*.png"))

if len(existing_frames) > 1000:
    print(f"\n2. Found {len(existing_frames)} existing frames, skipping extraction")
    frames = sorted(existing_frames)
else:
    print(f"\n2. Extracting {len(image_files)} images...")
    print("   This may take 5-10 minutes...\n")
    
    with zipfile.ZipFile(VIDEOS_PATH, 'r') as zf:
        for member in tqdm(image_files, desc="   Extracting"):
            zf.extract(member, DATA_DIR)
    
    # Find extracted frames
    frames = list(Path(DATA_DIR).rglob("*.jpg")) + list(Path(DATA_DIR).rglob("*.png"))
    frames = sorted(frames)

print(f"\n3. Verification:")
print(f"   Total frames found: {len(frames)}")

if len(frames) == 0:
    raise RuntimeError("No frames extracted! Check the ZIP file structure.")

# Show sample
print(f"   Sample frame: {frames[0]}")
print(f"\nExtraction successful!")

In [ ]:
#@title 6. Verify Images Can Be Loaded
from PIL import Image
import random

print("="*50)
print("VERIFYING IMAGES")
print("="*50)

# Test loading a few random images
test_frames = random.sample(frames, min(10, len(frames)))
valid_count = 0
errors = []

print(f"\nTesting {len(test_frames)} random images...")
for frame_path in test_frames:
    try:
        img = Image.open(frame_path)
        img.verify()  # Verify it's a valid image
        valid_count += 1
    except Exception as e:
        errors.append(f"{frame_path}: {e}")

print(f"Valid images: {valid_count}/{len(test_frames)}")

if errors:
    print(f"\nErrors found:")
    for err in errors[:5]:
        print(f"  - {err}")

if valid_count < len(test_frames) * 0.9:
    raise RuntimeError("Too many invalid images!")

# Show a sample image
sample_img = Image.open(frames[0])
print(f"\nSample image size: {sample_img.size}")
print(f"Sample image mode: {sample_img.mode}")
print("\nImages verified successfully!")

In [ ]:
#@title 7. Create SFT Dataset with Real Images
import numpy as np

print("="*50)
print("CREATING SFT DATASET")
print("="*50)

# Surgical vocabulary
PHASES = ["Nasal", "Sellar", "Tumor Removal", "Closure"]
STEPS = [
    "Septal Dissection", "Turbinectomy", "Sphenoidotomy", "Posterior Septectomy",
    "Sellar Floor Removal", "Dura Opening", "Tumor Resection", "Hemostasis",
    "Reconstruction", "Nasal Packing", "Visualization", "Instrument Change",
    "Suction", "Irrigation", "Other"
]
INSTRUMENTS = [
    "Endoscope", "Suction", "Curette", "Bipolar", "Monopolar", "Scissors",
    "Grasper", "Drill", "Kerrison", "Speculum", "Cottonoid", "Hemostatic Agent",
    "Fat Graft", "Fascia", "Nasoseptal Flap"
]

# Limit frames
frames_to_use = frames[:MAX_FRAMES]
print(f"\nUsing {len(frames_to_use)} frames (of {len(frames)} total)")
print(f"Will create {len(frames_to_use) * 3} samples (3 questions per frame)\n")

# Create samples with ACTUAL IMAGE PATHS
samples = []
for frame_path in tqdm(frames_to_use, desc="Creating samples"):
    frame_name = frame_path.name
    video_id = frame_path.parent.name
    
    # Random labels (in real use, these would come from annotations)
    phase = np.random.choice(PHASES)
    step = np.random.choice(STEPS)
    instruments = list(np.random.choice(INSTRUMENTS, np.random.randint(1, 4), replace=False))
    
    # Phase question
    samples.append({
        "messages": [
            {"role": "user", "content": "What surgical phase is shown in this image?"},
            {"role": "assistant", "content": f"This image shows the {phase} phase of pituitary surgery."}
        ],
        "image": str(frame_path),  # Full path - will be loaded by HFImage
        "video_id": video_id,
        "frame_id": frame_name,
        "qa_type": "phase"
    })
    
    # Step question
    samples.append({
        "messages": [
            {"role": "user", "content": "What surgical step is being performed?"},
            {"role": "assistant", "content": f"The surgeon is performing {step}."}
        ],
        "image": str(frame_path),
        "video_id": video_id,
        "frame_id": frame_name,
        "qa_type": "step"
    })
    
    # Instrument question
    if len(instruments) > 1:
        instr_str = ", ".join(instruments[:-1]) + f" and {instruments[-1]}"
    else:
        instr_str = instruments[0]
    
    samples.append({
        "messages": [
            {"role": "user", "content": "What surgical instruments are visible?"},
            {"role": "assistant", "content": f"The visible instruments are: {instr_str}."}
        ],
        "image": str(frame_path),
        "video_id": video_id,
        "frame_id": frame_name,
        "qa_type": "instruments"
    })

print(f"\nCreated {len(samples)} samples")
print(f"Sample keys: {list(samples[0].keys())}")

In [ ]:
#@title 8. Build HuggingFace Dataset WITH ACTUAL IMAGES
from datasets import Dataset, DatasetDict, Image as HFImage

print("="*50)
print("BUILDING HUGGINGFACE DATASET")
print("="*50)

print("\n1. Creating dataset from samples...")
dataset = Dataset.from_list(samples)
print(f"   Dataset size: {len(dataset)} rows")
print(f"   Columns: {dataset.column_names}")

print("\n2. Converting image paths to actual images...")
print("   This loads and encodes all images into the dataset.")
print("   This is the CRITICAL step that embeds real images!\n")

# THIS IS THE KEY LINE - it loads actual images from the file paths
dataset = dataset.cast_column("image", HFImage())

print(f"   Image column type: {dataset.features['image']}")

# Verify an image was loaded
print("\n3. Verifying images are embedded...")
sample = dataset[0]
if hasattr(sample['image'], 'size'):
    print(f"   Sample image size: {sample['image'].size}")
    print(f"   Sample image mode: {sample['image'].mode}")
    print("   Images are properly embedded!")
else:
    print(f"   WARNING: Image type is {type(sample['image'])}")

print("\n4. Splitting dataset...")
split = dataset.train_test_split(test_size=0.2, seed=42)
val_test = split["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

print(f"\nDataset splits:")
print(f"   Train: {len(dataset_dict['train'])} samples")
print(f"   Validation: {len(dataset_dict['validation'])} samples")
print(f"   Test: {len(dataset_dict['test'])} samples")
print("\nDataset ready for upload!")

In [ ]:
#@title 9. Push to HuggingFace Hub (30-60 min)
from huggingface_hub import create_repo, HfApi

print("="*50)
print("PUSHING TO HUGGINGFACE HUB")
print("="*50)

print(f"\nDataset: {DATASET_NAME}")
print(f"Total samples: {len(samples)}")
print("\nThis will upload all images and may take 30-60 minutes...\n")

# Create repo if needed
try:
    create_repo(DATASET_NAME, repo_type="dataset", exist_ok=True)
    print(f"Repository ready: {DATASET_NAME}")
except Exception as e:
    print(f"Note: {e}")

# Push dataset
print("\nUploading dataset with images...")
dataset_dict.push_to_hub(
    DATASET_NAME,
    private=False,
    commit_message="PitVQA SFT dataset with embedded surgical frame images"
)

print("\n" + "="*50)
print("SUCCESS!")
print("="*50)
print(f"\nDataset URL: https://huggingface.co/datasets/{DATASET_NAME}")

In [ ]:
#@title 10. Verify Upload
from datasets import load_dataset

print("="*50)
print("VERIFYING UPLOAD")
print("="*50)

print(f"\nLoading dataset from Hub: {DATASET_NAME}")
try:
    # Load just a few samples to verify
    verify_ds = load_dataset(DATASET_NAME, split="train[:5]")
    
    print(f"\nDataset loaded successfully!")
    print(f"Columns: {verify_ds.column_names}")
    print(f"Features: {verify_ds.features}")
    
    # Check if images are real
    sample = verify_ds[0]
    if hasattr(sample['image'], 'size'):
        print(f"\nImage verification:")
        print(f"  Type: PIL Image")
        print(f"  Size: {sample['image'].size}")
        print(f"  Mode: {sample['image'].mode}")
        print("\n" + "="*50)
        print("VERIFICATION PASSED - IMAGES ARE EMBEDDED!")
        print("="*50)
    else:
        print(f"\nWARNING: Image is type {type(sample['image'])}")
        print("Images may not be properly embedded.")
        
except Exception as e:
    print(f"Verification failed: {e}")

In [ ]:
#@title 11. Summary & Next Steps

print(f"""
{'='*60}
PIPELINE COMPLETE!
{'='*60}

Dataset: https://huggingface.co/datasets/{DATASET_NAME}

Statistics:
  - Train samples: {len(dataset_dict['train'])}
  - Validation samples: {len(dataset_dict['validation'])}
  - Test samples: {len(dataset_dict['test'])}
  - Total: {len(samples)}
  - Images: EMBEDDED (not just filenames!)

{'='*60}
NEXT STEP: Fine-tune SAGE/Molmo on this dataset
{'='*60}

Use this prompt in Claude Code with HF Skills:

Fine-tune allenai/SAGE-MM-Molmo2-8B-SFT_RL on {DATASET_NAME}

Configuration:
- Output model: {HF_USERNAME}/pitvqa-sage-surgical
- Epochs: 3
- Batch size: 4
- Learning rate: 2e-5
- Use LoRA: True (r=16, alpha=32)
- Hardware: a10g-large
- Vision language model with 'image' and 'messages' columns

{'='*60}
""")